# Stage 4: Main model (LightGBM)

This notebook builds the main shot make model, gradient boosted trees (LightGBM), and compares it with the best
baseline of Stage 3: the spline logistic regression trained on the last season (`logit_w1`, log loss 0.6402).

The validation is the same as in Stage 3: walk-forward by season, test seasons 2015-16 to 2024-25, hold-out season
2025-26 never loaded. Shared code: `xpts.validation`, `xpts.metrics`, `xpts.baselines`, `xpts.gbm`.

## 0. Setup

In [1]:
%load_ext autoreload
# re-import src/xpts after every change, no kernel restart needed
%autoreload 2

import numpy as np
import pandas as pd

from xpts.baselines import make_logit
from xpts.gbm import DEFAULT_PARAMS, GBM_FEATURES, GBMModel
from xpts.metrics import evaluate
from xpts.validation import HOLDOUT_SEASON, walk_forward

df = pd.read_parquet("../data/processed/shots_features.parquet")
df = df[df["SEASON"] != HOLDOUT_SEASON].reset_index(drop=True)  # hold-out season, never used here
y = df["SHOT_MADE_FLAG"]
print(f"shots: {len(df):,}, seasons: {df['SEASON'].min()} to {df['SEASON'].max()}")

shots: 2,305,805, seasons: 2014-15 to 2024-25


## 1. First LightGBM with default parameters

**Model.** `GBMModel` (in `src/xpts/gbm.py`) is a `LGBMClassifier` on the row-wise features of Stage 2:
distance, angle and raw coordinates, the 3PT / corner / heave / overtime flags, period and seconds left,
and `action_group` as a categorical feature. Unlike the logistic regression, the trees find non-linear effects and
interactions (e.g. distance x shot type x angle) by themselves, without splines.

**Parameters** are fixed by hand, not tuned yet (step 3): 300 trees, learning rate 0.05, 31 leaves, at least 200
shots per leaf, 80% of rows and columns per tree. No early stopping, because stopping on the test season would
leak it into the model.

Both models use the 1-season window (the best choice for the logit in Stage 3), so they see exactly the same data.

In [2]:
print(GBM_FEATURES)
print(DEFAULT_PARAMS)

models = {"logit_w1": make_logit, "gbm_w1": GBMModel}
oos = pd.DataFrame({"SEASON": df["SEASON"], "y": y})
rows = []
for name, make in models.items():
    for season, tr, te in walk_forward(df["SEASON"], window=1):
        p = make().fit(df[tr], y[tr]).predict_proba(df[te])[:, 1]
        oos.loc[te, name] = p
        rows.append({"model": name, "season": season, **evaluate(y[te], p)})

results = pd.DataFrame(rows)
oos = oos.dropna(subset=["logit_w1"])  # 2014-15 is only a training season
bias = oos.groupby("SEASON")[list(models)].mean().rsub(oos.groupby("SEASON")["y"].mean(), axis=0)
display(results.groupby("model")[["log_loss", "brier", "auc"]].mean().loc[list(models)].assign(bias=bias.mean()).round(4))
display(results.pivot(index="season", columns="model", values="log_loss")[list(models)].round(4), bias.round(3))

['dist_ft', 'angle_deg', 'LOC_X', 'LOC_Y', 'is_three', 'is_corner3', 'is_heave', 'is_overtime', 'PERIOD', 'secs_left_period', 'action_group']
{'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 200, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'verbose': -1}


,log_loss,brier,auc,bias
model,,,,
logit_w1,0.6402,0.2255,0.6562,0.0007
gbm_w1,0.6413,0.2260,0.6537,0.0018


model,logit_w1,gbm_w1
season,,
2015-16,0.6367,0.6368
2016-17,0.6386,0.6374
2017-18,0.6478,0.6570
2018-19,0.6372,0.6370
2019-20,0.6367,0.6365
2020-21,0.6446,0.6466
2021-22,0.6395,0.6400
2022-23,0.6401,0.6402
2023-24,0.6406,0.6406


,logit_w1,gbm_w1
SEASON,,
2015-16,-0.012,-0.010
2016-17,0.001,0.002
2017-18,-0.003,0.009
2018-19,-0.006,-0.006
2019-20,0.004,0.004
2020-21,0.016,0.012
2021-22,-0.002,-0.002
2022-23,0.011,0.011
2023-24,0.001,0.001


**Observations.**
- The untuned LightGBM is **not better** than the logistic regression: mean log loss 0.6413 vs 0.6402, AUC 0.654 vs 0.656.
- In 7 of 10 seasons the two are within ±0.001; LightGBM is better in 2016-17 and clearly worse in 2017-18
  (0.657 vs 0.648) and 2020-21. 2017-18 is the season in which the label vocabulary changed (Stage 3, section 5b):
  the trees fitted on 2016-17 split on label patterns that no longer exist.
- So with these features the spline logit already captures most of the signal. Next steps: the training window
  (more seasons give the trees more data, but older labels), then hyperparameter tuning inside the training window.

## 2. Training window: short vs expanding

**Question.** In Stage 3 the logit was best with a 1-season window, because older seasons use the old `ACTION_TYPE`
labels. Trees need more data than a logit, so a longer window could help LightGBM even with the label drift.
We compare windows of 1, 2 and 3 seasons and the expanding window (all previous seasons), same walk-forward test seasons.

In [ ]:
windows = {"gbm_w1": 1, "gbm_w2": 2, "gbm_w3": 3, "gbm_exp": None}
rows_w = [r for r in rows if r["model"] in ("logit_w1", "gbm_w1")]
for name, w in windows.items():
    if name == "gbm_w1":
        continue  # already in section 1
    for season, tr, te in walk_forward(df["SEASON"], window=w):
        p = GBMModel().fit(df[tr], y[tr]).predict_proba(df[te])[:, 1]
        oos.loc[te, name] = p
        rows_w.append({"model": name, "season": season, **evaluate(y[te], p)})

res_w = pd.DataFrame(rows_w)
order = ["logit_w1", *windows]
display(res_w.groupby("model")[["log_loss", "brier", "auc"]].mean().loc[order].round(4))
display(res_w.pivot(index="season", columns="model", values="log_loss")[order].round(4))